In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import wandb
from huggingface_hub import HfApi, login
import matplotlib.pyplot as plt
import numpy as np
import time
from tqdm.auto import tqdm
import os
import pandas as pd
from IPython.display import display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

Using device: cuda
Tesla T4


In [2]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kumarmayank_25afi14 (kumarmayank_25afi14-dtu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
from google.colab import userdata
try:
    hf_token = userdata.get('')
    login(token=hf_token)
except:
    print("HF_TOKEN secret not found. Please login interactively:")
    login()

HF_TOKEN secret not found. Please login interactively:


In [4]:
batch_size = 256
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) 
])
train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
print(f"Training dataset size: {len(train_data)}")

100%|██████████| 26.4M/26.4M [00:02<00:00, 13.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 209kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.90MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.2MB/s]

Training dataset size: 60000


In [5]:
latent_dim = 100
class VanillaGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 784),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        return img.view(img.size(0), 1, 28, 28)

class VanillaDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 1)
        )

    def forward(self, img):
        img_flat = img.view(img.size(0), -1)
        return self.model(img_flat)
        
class DCGANGenerator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 256, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )

    def forward(self, z):
        z = z.view(z.size(0), latent_dim, 1, 1)
        return self.model(z)

class DCGANDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 128, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Flatten(),
            nn.Linear(256 * 7 * 7, 1)
        )

    def forward(self, img):
        return self.model(img)

In [7]:
def calculate_loss(real_logits, fake_logits, loss_type):
    if loss_type == "BCE":
        real_loss = F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits))
        fake_loss = F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))
        d_loss = (real_loss + fake_loss) / 2
        g_loss = F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))
        
    elif loss_type == "LSGAN":
        real_loss = F.mse_loss(real_logits, torch.ones_like(real_logits))
        fake_loss = F.mse_loss(fake_logits, torch.zeros_like(fake_logits))
        d_loss = 0.5 * (real_loss + fake_loss)
        g_loss = 0.5 * F.mse_loss(fake_logits, torch.ones_like(fake_logits))
        
    elif loss_type == "WGAN":
        d_loss = -torch.mean(real_logits) + torch.mean(fake_logits)
        g_loss = -torch.mean(fake_logits)
        
    return d_loss, g_loss

In [9]:
def train(arch_type, loss_type, opt_name, epochs=20):
    run = wandb.init(
        project="dl-lab-9-gan",
        name=f"{arch_type}_{loss_type}_{opt_name}",
        config={"architecture": arch_type, "loss": loss_type, "optimizer": opt_name},
    )
    if arch_type == "Vanilla":
        generator = VanillaGenerator().to(device)
        discriminator = VanillaDiscriminator().to(device)
    else:
        generator = DCGANGenerator().to(device)
        discriminator = DCGANDiscriminator().to(device)

    lr = 2e-4
    if opt_name == "Adam":
        optimizer_gen = optim.Adam(generator.parameters(), lr=lr, betas=(0.0, 0.999))
        optimizer_disc = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.0, 0.999))
    elif opt_name == "RMSprop":
        optimizer_gen = optim.RMSprop(generator.parameters(), lr=lr, momentum=0.0)
        optimizer_disc = optim.RMSprop(discriminator.parameters(), lr=lr, momentum=0.0)
    elif opt_name == "SGD":
        optimizer_gen = optim.SGD(generator.parameters(), lr=lr, momentum=0.0)
        optimizer_disc = optim.SGD(discriminator.parameters(), lr=lr, momentum=0.0)

    fixed_noise = torch.randn(8, latent_dim).to(device)

    print(f"{arch_type}_{loss_type}_{opt_name}")
    for epoch in range(epochs):
        generator.train()
        discriminator.train()
        
        d_loss_epoch = 0
        g_loss_epoch = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for real_imgs, _ in pbar:
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)
            
            optimizer_disc.zero_grad()
            z = torch.randn(batch_size, latent_dim).to(device)
            fake_imgs = generator(z)
            real_logits = discriminator(real_imgs)
            fake_logits = discriminator(fake_imgs.detach())
            d_loss, _ = calculate_loss(real_logits, fake_logits, loss_type)
            d_loss.backward()
            optimizer_disc.step()
            if loss_type == "WGAN":
                for p in discriminator.parameters():
                    p.data.clamp_(-0.01, 0.01)

            optimizer_gen.zero_grad()
            fake_logits_for_G = discriminator(fake_imgs)
            _, g_loss = calculate_loss(real_logits, fake_logits_for_G, loss_type)
            g_loss.backward()
            optimizer_gen.step()
            d_loss_epoch += d_loss.item()
            g_loss_epoch += g_loss.item()
            pbar.set_postfix({'D-Loss': f"{d_loss.item():.4f}", 'G-Loss': f"{g_loss.item():.4f}"})
            
        avg_d_loss = d_loss_epoch / len(train_loader)
        avg_g_loss = g_loss_epoch / len(train_loader)
        log_data = {
            "discriminator_loss": avg_d_loss,
            "generator_loss": avg_g_loss
        }

        if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
            generator.eval()
            with torch.no_grad():
                gen_samples = generator(fixed_noise).cpu()
            gen_samples = (gen_samples + 1) / 2.0 
            
            fig, axes = plt.subplots(2, 4, figsize=(6, 4))
            for i, ax in enumerate(axes.flatten()):
                ax.imshow(gen_samples[i].squeeze(), cmap='gray')
                ax.axis('off')

            log_data["generated_samples"] = wandb.Image(fig)
            plt.close(fig)
        
        wandb.log(log_data)

    run.finish()
    return generator, avg_d_loss, avg_g_loss

In [ ]:
models = ['Vanilla','DCGAN']
losses = ['BCE', 'LSGAN', 'WGAN']
optimizers = ['Adam', 'RMSprop', 'SGD']

trained_generators = {}
results = []

for model in models: 
    for loss_fn in losses: 
        for opt in optimizers:
            gen_model, final_d_loss, final_g_loss = train(model, loss_fn, opt, epochs=50) 
            results.append({
                "Model": model,
                "Loss Fn": loss_fn,
                "Optimizer": opt,
                "Final D-Loss": round(final_d_loss, 4),
                "Final G-Loss": round(final_g_loss, 4)
            })
            model_name = f"Generator_{model}_{loss_fn}_{opt}"
            trained_generators[model_name] = gen_model

Vanilla_BCE_Adam


Epoch 1/50:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 2/50:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/235 [00:00<?, ?it/s]

...

Epoch 2/50:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 3/50:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 4/50:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 5/50:   0%|          | 0/235 [00:00<?, ?it/s]

Epoch 6/50:   0%|          | 0/235 [00:00<?, ?it/s]

In [1]:
df_gan_results = pd.DataFrame(results)
display(df_gan_results)

,Model,Loss Fn,Optimizer,Final D-Loss,Final G-Loss
0,Vanilla,BCE,Adam,0.6407,0.9306
1,Vanilla,BCE,RMSprop,0.6310,0.9510
2,Vanilla,BCE,SGD,0.3281,0.7835
3,Vanilla,LSGAN,Adam,0.1985,0.2376
4,Vanilla,LSGAN,RMSprop,0.1727,0.2789
5,Vanilla,LSGAN,SGD,0.0080,0.4115
6,Vanilla,WGAN,Adam,-0.0198,-0.1818
7,Vanilla,WGAN,RMSprop,-0.0197,-0.1215
8,Vanilla,WGAN,SGD,-1.9219,-0.0309
9,DCGAN,BCE,Adam,0.4062,2.0955


In [ ]:
repo_id = "MK80310/DLLab9" 

api = HfApi()

try:
    api.create_repo(repo_id=repo_id, exist_ok=True)
    print(f"Repository {repo_id} ready.")
    
    for name, model in trained_generators.items():
        save_path = f"{name}.pth"
        torch.save(model.state_dict(), save_path)
        
        api.upload_file(
            path_or_fileobj=save_path,
            path_in_repo=f"models/{save_path}",
            repo_id=repo_id,
            commit_message=f"Add {name} generator weights"
        )
        
    print("Models successfully pushed to Hugging Face Hub!")
    
except Exception as e:
    print(f"Error during upload: {e}")